<a href="https://colab.research.google.com/github/jieyingc/fast-failure-detection/blob/main/Week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset

dataset = load_dataset("SWE-bench/SWE-smith-trajectories")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/tool-00000-of-00008.parquet:   0%|          | 0.00/249M [00:00<?, ?B/s]

data/tool-00001-of-00008.parquet:   0%|          | 0.00/116M [00:00<?, ?B/s]

data/tool-00002-of-00008.parquet:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

...

data/ticks-00007-of-00008.parquet:   0%|          | 0.00/229M [00:00<?, ?B/s]

Generating tool split:   0%|          | 0/24100 [00:00<?, ? examples/s]

Generating xml split:   0%|          | 0/26076 [00:00<?, ? examples/s]

Generating ticks split:   0%|          | 0/25826 [00:00<?, ? examples/s]

DatasetDict({
    tool: Dataset({
        features: ['messages', 'instance_id', 'resolved', 'model', 'traj_id', 'patch'],
        num_rows: 24100
    })
    xml: Dataset({
        features: ['messages', 'instance_id', 'resolved', 'model', 'traj_id', 'patch'],
        num_rows: 26076
    })
    ticks: Dataset({
        features: ['messages', 'instance_id', 'resolved', 'model', 'traj_id', 'patch'],
        num_rows: 25826
    })
})


#Investigate Empty Patch

1. Nearly half of empty patch are resolved.

2. In empty patch, 1/3 of both resolved and unresolved have "submit" in last message action. Only difference is False action = ‘’ slightly higher.

3. Similar average bash and edit calls.

4. 97% of empty patch trajectory have been edited.

Empty patch field does not necessarily mean that no code modification occurred. The empty patch is excluded in the later model training.

In [2]:
from collections import Counter

for split in dataset.keys():
    data = dataset[split]

    empty = []
    non_empty = []

    for row in data:
        if row["patch"].strip() == "":
            empty.append(row["resolved"])
        else:
            non_empty.append(row["resolved"])

    print("====", split, "====")
    print("empty patch count:", len(empty))
    print("non-empty patch count:", len(non_empty))

    print("empty patch resolved distribution:", Counter(empty))
    print(f"True %: {(Counter(empty)[True]/sum(Counter(empty).values())) * 100:.2f}%")

    print("non-empty patch resolved distribution:", Counter(non_empty))
    print(f"True %: {(Counter(non_empty)[True]/sum(Counter(non_empty).values())) * 100:.2f}%")

    print()

==== tool ====
empty patch count: 5944
non-empty patch count: 18156
empty patch resolved distribution: Counter({False: 3662, True: 2282})
True %: 38.39%
non-empty patch resolved distribution: Counter({False: 11011, True: 7145})
True %: 39.35%

==== xml ====
empty patch count: 7352
non-empty patch count: 18724
empty patch resolved distribution: Counter({False: 4029, True: 3323})
True %: 45.20%
non-empty patch resolved distribution: Counter({False: 10544, True: 8180})
True %: 43.69%

==== ticks ====
empty patch count: 7279
non-empty patch count: 18547
empty patch resolved distribution: Counter({False: 3983, True: 3296})
True %: 45.28%
non-empty patch resolved distribution: Counter({False: 10413, True: 8134})
True %: 43.86%



In [3]:
import json

True_actions = Counter()

for row in dataset["tool"]:
    if row["patch"].strip() == "" and row["resolved"]:
        msgs = json.loads(row["messages"])

        last_msg = msgs[-1]

        action = last_msg.get("action", "none")

        True_actions[action] += 1

print("Empty patch Last message action distribution (Resolved = True):", True_actions)

False_actions = Counter()

for row in dataset["tool"]:
    if row["patch"].strip() == "" and not row["resolved"]:
        msgs = json.loads(row["messages"])

        last_msg = msgs[-1]

        action = last_msg.get("action", "none")

        False_actions[action] += 1

print("Empty patch Last message action distribution (Resolved = False):", False_actions)

Empty patch Last message action distribution (Resolved = True): Counter({'none': 1510, 'submit': 745, '': 27})
Empty patch Last message action distribution (Resolved = False): Counter({'none': 2245, 'submit': 1218, '': 199})


In [4]:
import numpy as np

empty_bash = []
nonempty_bash = []

empty_edit = []
nonempty_edit = []

for row in dataset["tool"]:
    msgs = json.loads(row["messages"])

    bash_calls = 0
    edit_calls = 0

    for msg in msgs:
        tool_calls = msg.get("tool_calls") or []

        for tc in tool_calls:
            fn = tc.get("function", {}).get("name", "")

            if fn == "bash":
                bash_calls += 1
            if fn == "str_replace_editor":
                edit_calls += 1

    if row["patch"].strip() == "":
        empty_bash.append(bash_calls)
        empty_edit.append(edit_calls)
    else:
        nonempty_bash.append(bash_calls)
        nonempty_edit.append(edit_calls)

print(f"avg bash calls (empty patch): {np.mean(empty_bash):.2f}")
print(f"avg bash calls (non-empty patch): {np.mean(nonempty_bash):.2f}")

print(f"avg edit calls (empty patch): {np.mean(empty_edit):.2f}")
print(f"avg edit calls (non-empty patch): {np.mean(nonempty_edit):.2f}")

avg bash calls (empty patch): 4.29
avg bash calls (non-empty patch): 4.49
avg edit calls (empty patch): 6.28
avg edit calls (non-empty patch): 6.54


In [5]:
for row in dataset["tool"]:
    if row["patch"].strip() == "" and row["resolved"]:

        msgs = json.loads(row["messages"])

        print("INSTANCE:", row["instance_id"])
        print("RESOLVED:", row["resolved"])
        print("NUM_MESSAGES:", len(msgs))

        print("\nLast 5 messages:")
        for m in msgs[-5:]:
            print(m)

        break

INSTANCE: django-money__django-money.835c1ab8.func_pm_ctrl_shuffle__viqnyl9u
RESOLVED: True
NUM_MESSAGES: 49

Last 5 messages:
{'role': 'assistant', 'content': "Great! The form with a non-required field is valid, which is the expected behavior. This confirms that our fix works correctly for all cases.\n\n## Summary\n\nWe've successfully fixed the issue with the `MoneyWidget` class in the forms/widgets.py file. The problem was in the `decompress` method, which was immediately returning `[None, self.default_currency]` regardless of the input value. This was causing validation to fail for disabled fields.\n\nOur fix was to reorder the conditional statements in the `decompress` method so that it first checks if `value is not None` before returning the appropriate result. This ensures that the method correctly handles Money values, list/tuple values, and None values.\n\nWe've tested the fix with various edge cases, including:\n1. A form with a disabled field and an instance\n2. A form with 

In [6]:
stats = {
    True: {"total": 0, "with_diff": 0, "with_edited": 0, "with_review": 0},
    False: {"total": 0, "with_diff": 0, "with_edited": 0, "with_review": 0},
}

for row in dataset["tool"]:
    if row["patch"].strip() != "":
        continue

    label = row["resolved"]
    stats[label]["total"] += 1

    msgs = json.loads(row["messages"])
    full_text = json.dumps(msgs).lower()

    if "diff --git" in full_text:           # diff information distributed across user/tool messages
        stats[label]["with_diff"] += 1

    if "has been edited" in full_text:
        stats[label]["with_edited"] += 1

    if "here is a list of all of your changes" in full_text:
        stats[label]["with_review"] += 1

for label in [True, False]:
    s = stats[label]
    print("resolved =", label)
    print("total:", s["total"])
    print(f"with_diff: {s["with_diff"]}, {s["with_diff"] / s["total"]:.2f}")
    print(f"with_edited: {s["with_edited"]}, {s["with_edited"] / s["total"]:.2f}")
    print(f"with_review: {s["with_review"]}, {s["with_review"] / s["total"]:.2f}")
    print()

resolved = True
total: 2282
with_diff: 1421, 0.62
with_edited: 2230, 0.98
with_review: 1362, 0.60

resolved = False
total: 3662
with_diff: 2272, 0.62
with_edited: 3539, 0.97
with_review: 2187, 0.60



#Split by Instance_ids